# Grafo de conocimiento integrando distintas fuentes

## Fuentes de Integración

Integra datos de Wikidata con ontologías musicales externas:
- Wikidata: Datos estructurados vía SPARQL
- MusicBrainz REST API: Discografía detallada (JSON→RDF)
- Music Ontology: Vocabulario formal
- Schema.org: Propiedades estándar



Importar Librerías

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
from rdflib.namespace import FOAF, DCTERMS
import requests
import json
import time
from datetime import datetime
import sbc_tools as sbc

## Inicialización del grafo de conocimiento

Crear grafo principal y definir todos los namespaces necesarios

In [2]:
graph = Graph()

WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
MO = Namespace("http://purl.org/ontology/mo/")
SCHEMA = Namespace("https://schema.org/")
MB = Namespace("https://musicbrainz.org/")
LOCAL = Namespace("http://informatica.ucm.es/musicontology/")

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(SPARQL_JSON)

graph.bind("wd", WD)
graph.bind("wdt", WDT)
graph.bind("mo", MO)
graph.bind("schema", SCHEMA)
graph.bind("mb", MB)
graph.bind("local", LOCAL)
graph.bind("foaf", FOAF)
graph.bind("dcterms", DCTERMS)
graph.bind("owl", OWL)
graph.bind("rdfs", RDFS)

---

# Crear Jerarquía de Clases Unificada

## Crea jerarquía de clases unificando diferentes ontologías

### Declarar ontología base

In [3]:
graph.add((LOCAL[""], RDF.type, OWL.Ontology))
graph.add((LOCAL[""], RDFS.label, Literal("Ontología Musical Unificada")))
graph.add((LOCAL[""], RDFS.comment, Literal("Integración de Wikidata, Music Ontology y Schema.org")))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

## CLASES BASE

### Artista Musical (unifica conceptos de diferentes ontologías)

In [4]:
graph.add((LOCAL.MusicalArtist, RDF.type, OWL.Class))
graph.add((LOCAL.MusicalArtist, RDFS.label, Literal("Artista Musical")))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Mapear a ontologías externas (owl:equivalentClass)

In [5]:
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, SCHEMA.MusicGroup))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.MusicalArtist, OWL.equivalentClass, SCHEMA.MusicGroup))


<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Subclases de Artista Musical (ejemplo)

In [6]:
artist_types = [
    ("SoloArtist", "Artista en solitario", MO.SoloMusicArtist),
    ("Band", "Banda musical", MO.MusicGroup)
]
        
for class_name, label, external_class in artist_types:
    class_uri = LOCAL[class_name]
    graph.add((class_uri, RDF.type, OWL.Class))
    graph.add((class_uri, RDFS.subClassOf, LOCAL.MusicalArtist))
    graph.add((class_uri, RDFS.label, Literal(label)))
    if external_class:
        graph.add((class_uri, OWL.equivalentClass, external_class))

## OBRAS MUSICALES

In [7]:
graph.add((LOCAL.MusicalWork, RDF.type, OWL.Class))
graph.add((LOCAL.MusicalWork, RDFS.label, Literal("Obra Musical")))
graph.add((LOCAL.MusicalWork, OWL.equivalentClass, MO.MusicalWork))
graph.add((LOCAL.MusicalWork, OWL.equivalentClass, SCHEMA.MusicComposition))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Subclases de Obras Musicales (ejemplo)

In [8]:
work_types = [
    ("Album", "Álbum", MO.Record, SCHEMA.MusicAlbum),
    ("Song", "Canción", MO.Track, SCHEMA.MusicRecording)
]

for class_name, label, mo_class, schema_class in work_types:
    class_uri = LOCAL[class_name]
    graph.add((class_uri, RDF.type, OWL.Class))
    graph.add((class_uri, RDFS.subClassOf, LOCAL.MusicalWork))
    graph.add((class_uri, RDFS.label, Literal(label)))
    if mo_class:
        graph.add((class_uri, OWL.equivalentClass, mo_class))
    if schema_class:
        graph.add((class_uri, OWL.equivalentClass, schema_class))

## GÉNEROS MUSICALES

In [9]:
graph.add((LOCAL.MusicGenre, RDF.type, OWL.Class))
graph.add((LOCAL.MusicGenre, RDFS.label, Literal("Género Musical")))
graph.add((LOCAL.MusicGenre, OWL.equivalentClass, MO.Genre))
graph.add((LOCAL.MusicGenre, OWL.equivalentClass, SCHEMA.MusicGenre))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

Usaremos el código de la práctica anterior para visualizar la ontología creada

In [10]:
sbc.show_graph(graph, output_file="grafo_ontologia_musical_jerarquia.html")

out/grafo_ontologia_musical_jerarquia.html


---

# Crear Propiedades Unificadas

## Define propiedades unificadas mapeando a ontologías externas

## PROPIEDADES DE OBJETO

### Propiedad: creador/artista

In [11]:
graph.add((LOCAL.creator, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.creator, RDFS.label, Literal("creador")))
graph.add((LOCAL.creator, OWL.equivalentProperty, MO.composer))
graph.add((LOCAL.creator, OWL.equivalentProperty, SCHEMA.byArtist))
graph.add((LOCAL.creator, OWL.equivalentProperty, DCTERMS.creator))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Propiedad: género musical

In [12]:
graph.add((LOCAL.hasGenre, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.hasGenre, RDFS.label, Literal("tiene género")))
graph.add((LOCAL.hasGenre, RDFS.domain, LOCAL.MusicalWork))
graph.add((LOCAL.hasGenre, RDFS.range, LOCAL.MusicGenre))
graph.add((LOCAL.hasGenre, OWL.equivalentProperty, MO.genre))
graph.add((LOCAL.hasGenre, OWL.equivalentProperty, SCHEMA.genre))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Propiedad: miembro de banda

In [13]:
graph.add((LOCAL.memberOf, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.memberOf, RDFS.label, Literal("miembro de")))
graph.add((LOCAL.memberOf, RDFS.domain, LOCAL.SoloArtist))
graph.add((LOCAL.memberOf, RDFS.range, LOCAL.Band))
graph.add((LOCAL.memberOf, OWL.equivalentProperty, MO.member_of))
graph.add((LOCAL.memberOf, OWL.equivalentProperty, SCHEMA.memberOf))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

## PROPIEDADES DE DATOS

### Fecha de publicación

In [14]:
graph.add((LOCAL.releaseDate, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.releaseDate, RDFS.label, Literal("fecha de lanzamiento")))
graph.add((LOCAL.releaseDate, RDFS.domain, LOCAL.MusicalWork))
graph.add((LOCAL.releaseDate, RDFS.range, XSD.date))
graph.add((LOCAL.releaseDate, OWL.equivalentProperty, SCHEMA.datePublished))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Duración

In [15]:
graph.add((LOCAL.duration, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.duration, RDFS.label, Literal("duración")))
graph.add((LOCAL.duration, RDFS.domain, LOCAL.Song))
graph.add((LOCAL.duration, RDFS.range, XSD.duration))
graph.add((LOCAL.duration, OWL.equivalentProperty, MO.duration))
graph.add((LOCAL.duration, OWL.equivalentProperty, SCHEMA.duration))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### MusicBraind ID

In [16]:
graph.add((LOCAL.mbid, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.mbid, RDFS.label, Literal("MusicBrainz ID")))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

### Popularidad

In [17]:
graph.add((LOCAL.popularity, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.popularity, RDFS.label, Literal("popularidad")))
graph.add((LOCAL.popularity, RDFS.range, XSD.integer))

<Graph identifier=Ncae3157cda5d4a2b87779986ea6296f9 (<class 'rdflib.graph.Graph'>)>

---

# Obtener artistas desde Wikidata

## Obtiene artistas musicales de Wikidata

**Parámetros:**
- `genre_name`: Género musical a buscar
- `limit`: Número máximo de resultados

**Consulta SPARQL:**
- Busca humanos (Q5) que sean músicos (Q639669)
- Filtra por género musical especificado
- Obtiene información adicional: país, fecha de nacimiento


In [18]:
def get_artists_from_wikidata(genre_name="pop music", limit=10):

    query = f"""
        SELECT DISTINCT ?artist ?artistLabel ?genreLabel ?countryLabel ?birthDate ?mbid WHERE {{
        ?artist wdt:P31 wd:Q5 ;
                wdt:P106 wd:Q639669 ;
                wdt:P136 ?genre .

        ?artist rdfs:label ?artistLabel .
        ?genre rdfs:label ?genreLabel .
        FILTER (lang(?genreLabel) = 'en')
        FILTER(CONTAINS(LCASE(?genreLabel), "{genre_name.lower()}"))

        OPTIONAL {{ ?artist wdt:P27 ?country . 
                    ?country rdfs:label ?countryLabel
                    FILTER (lang(?countryLabel) = 'es')
                }}
        OPTIONAL {{ ?artist wdt:P569 ?birthDate . }}
        OPTIONAL {{ ?artist wdt:P434 ?mbid . }}
        FILTER (lang(?artistLabel) = 'es')
        

        }}
        LIMIT {limit}
        """

    sparql.setQuery(query)
    results = sparql.query().convert()

    artists_data = []
    for result in results["results"]["bindings"]:
        artists_data.append({
            'uri': result["artist"]["value"],
            'name': result["artistLabel"]["value"],
            'genre': result.get("genreLabel", {}).get("value", "Desconocido"),
            'country': result.get("countryLabel", {}).get("value", "Desconocido"),
            'birthDate': result.get("birthDate", {}).get("value", None),
            'mbid': result.get("mbid", {}).get("value", None)
        })
    
    return artists_data

In [19]:
# lo probamos
artists = get_artists_from_wikidata(genre_name="pop music", limit=5)
for artist in artists:
    print(artist)

{'uri': 'http://www.wikidata.org/entity/Q48737', 'name': 'Myleene Klass', 'genre': 'pop music', 'country': 'Reino Unido', 'birthDate': '1978-04-06T00:00:00Z', 'mbid': '060f0ad4-9c6e-41a9-90fd-355636641971'}
{'uri': 'http://www.wikidata.org/entity/Q5383', 'name': 'David Bowie', 'genre': 'pop music', 'country': 'Reino Unido', 'birthDate': '1947-01-08T00:00:00Z', 'mbid': '5441c29d-3602-4898-b1a1-b77fa23b8e50'}
{'uri': 'http://www.wikidata.org/entity/Q66771', 'name': 'Jasmin Tabatabai', 'genre': 'pop music', 'country': 'Irán', 'birthDate': '1967-06-08T00:00:00Z', 'mbid': 'fd214d5a-8a2d-4f16-b430-cebe4ea682fc'}
{'uri': 'http://www.wikidata.org/entity/Q63046', 'name': 'Corinna May', 'genre': 'pop music', 'country': 'Alemania', 'birthDate': '1970-10-10T00:00:00Z', 'mbid': '688d73ef-ec40-482b-bfac-d927e43fde03'}
{'uri': 'http://www.wikidata.org/entity/Q58168', 'name': 'Nolwenn Leroy', 'genre': 'pop music', 'country': 'Francia', 'birthDate': '1982-09-28T00:00:00Z', 'mbid': '3575a514-ef96-41a0-a

---

# Integrar Artistas  de Wikidata

## Integra datos de artistas de Wikidata en la ontología unificada

**Para cada artista:**
1. Añade como instancia de la clase unificada
2. Mapea a clases de ontologías externas
3. Añade propiedades usando vocabularios estándar (FOAF, Schema.org)
4. Incluye información contextual (país, fecha de nacimiento, género)

In [25]:
def integrate_wikidata_artists(artists_data):
    
    for artist in artists_data:
        artist_uri = URIRef(artist['uri'])

        graph.add((artist_uri, RDF.type, LOCAL.SoloArtist))
        graph.add((artist_uri, RDFS.label, Literal(artist['name'], lang="es")))

        graph.add((artist_uri, RDF.type, MO.MusicArtist))
        graph.add((artist_uri, RDF.type, SCHEMA.MusicGroup))

        graph.add((artist_uri, FOAF.name, Literal(artist['name'])))
        
        if artist['country'] != "Desconocido":
            graph.add((artist_uri, LOCAL.country, Literal(artist['country'])))
            graph.add((artist_uri, SCHEMA.nationality, Literal(artist['country'])))

        if artist['birthDate']:
            birth_date = artist['birthDate'][:10]
            graph.add((artist_uri, SCHEMA.birthDate,
                        Literal(birth_date, datatype=XSD.date)))
        
        if artist['genre']:
            genre_uri = LOCAL[artist['genre'].replace(" ", "_")]
            graph.add((genre_uri, RDF.type, LOCAL.MusicGenre))
            graph.add((genre_uri, RDFS.label, Literal(artist['genre'])))
            graph.add((artist_uri, LOCAL.hasGenre, genre_uri))


In [21]:
integrate_wikidata_artists(artists)

In [22]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_base.html")

out/grafo_ontologia_musical_base.html


---

# Enriquecer con MusicBrainz REST API

URL base

In [26]:
mb_base_url = "https://musicbrainz.org/ws/2/"
mb_headers = {
    'User-Agent': 'MusicOntologyIntegrator/1.0 (educational purpose)',
    'Accept': 'application/json'
}

## Buscar artista en MusicBrainz por nombre

Endpoint: https://musicbrainz.org/ws/2/artist/

In [27]:
def search_artist_musicbrainz(artist_name):
        
        url = f"{mb_base_url}artist/"
        params = {
            'query': artist_name,
            'fmt': 'json',
            'limit': 1
        }
        
        try:
            response = requests.get(url, params=params, headers=mb_headers)
            time.sleep(1)
            
            if response.status_code == 200:
                data = response.json()
                if data.get('artists') and len(data['artists']) > 0:
                    artist = data['artists'][0]
                    return {
                        'mbid': artist.get('id'),
                        'name': artist.get('name'),
                        'type': artist.get('type'),
                        'country': artist.get('country'),
                        'life_span': artist.get('life-span', {})
                    }
        except Exception as e:
            print(f"Error: {e}")
        
        return None

In [28]:
# lo probamos
search_artist_musicbrainz("Raphael")

{'mbid': 'b34cfe02-994d-40dc-854f-bf54a23dd405',
 'name': 'Raphael',
 'type': 'Person',
 'country': 'ES',
 'life_span': {'begin': '1943-05-05', 'ended': None}}

## Obtener releases (álbumes) del artista

Endpoint: https://musicbrainz.org/ws/2/release/

In [29]:
def get_releases_musicbrainz(mbid):
        
    if not mbid:
        return None
    
    url = f"{mb_base_url}release/"
    params = {
        'artist': mbid,
        'type': 'album',
        'fmt': 'json',
        'limit': 10
    }
    
    try:
        response = requests.get(url, params=params, headers=mb_headers)
        time.sleep(1)
        
        if response.status_code == 200:
            data = response.json()
            releases = []
            
            for release in data.get('releases', []):
                releases.append({
                    'mbid': release.get('id'),
                    'title': release.get('title'),
                    'date': release.get('date'),
                    'country': release.get('country'),
                    'status': release.get('status')
                })
            
            return releases
    except Exception as e:
        print(f"Error: {e}")
    
    return None

In [30]:
# lo probamos
get_releases_musicbrainz('b34cfe02-994d-40dc-854f-bf54a23dd405')  # MBID de Raphael

[{'mbid': '02501648-b75b-4dfc-a3b7-ae3c45a56b12',
  'title': 'El ángel',
  'date': '1969',
  'country': 'EC',
  'status': 'Official'},
 {'mbid': '26c5a48b-e40d-45a9-a0e2-7bdd25c5faa6',
  'title': 'El golfo',
  'date': '1968',
  'country': 'ES',
  'status': 'Official'},
 {'mbid': '2a3dac67-22ad-42d3-b459-a6a69cf56d2b',
  'title': 'Digan lo que digan',
  'date': '1967',
  'country': 'ES',
  'status': 'Official'},
 {'mbid': '4ca3895d-4bbf-4f48-8c13-4a87c6b599f9',
  'title': 'Más Raphael',
  'date': '1968',
  'country': 'AR',
  'status': 'Official'},
 {'mbid': '5493d567-71aa-43d0-ba41-b5530aecf7d0',
  'title': 'Raphael canta…',
  'date': '1966',
  'country': None,
  'status': None},
 {'mbid': '90040412-8f62-47f7-94ae-8294038b1070',
  'title': 'Al ponerse el sol',
  'date': '1967',
  'country': 'ES',
  'status': 'Official'},
 {'mbid': 'ab521d79-75d0-4a74-a2b8-f9e437e38a1c',
  'title': 'Aleluya...',
  'date': '1970',
  'country': 'ES',
  'status': 'Official'},
 {'mbid': 'b6d031a1-67dd-49cf-9

## Obtener recordings (canciones) de un release

Endpoint: https://musicbrainz.org/ws/2/release/{mbid}?inc=recordings

In [31]:
def get_recordings_from_release(release_mbid):
        
    if not release_mbid:
        return None

    url = f"{mb_base_url}release/{release_mbid}"
    params = {
        'inc': 'recordings',
        'fmt': 'json'
    }
    
    try:
        response = requests.get(url, params=params, headers=mb_headers)
        time.sleep(1)
        
        if response.status_code == 200:
            data = response.json()
            recordings = []
            
            for medium in data.get('media', []):
                for track in medium.get('tracks', []):
                    recording = track.get('recording', {})
                    recordings.append({
                        'mbid': recording.get('id'),
                        'title': recording.get('title'),
                        'length': recording.get('length')
                    })
            
            return recordings
    except Exception as e:
        print(f"Error: {e}")
    
    return None

In [32]:
# lo probamos
get_recordings_from_release('2a3dac67-22ad-42d3-b459-a6a69cf56d2b')

[{'mbid': '510752d8-7e03-484d-ba19-da2355d8aed8',
  'title': 'Digan lo que digan',
  'length': 164000},
 {'mbid': 'cbec12f6-15f6-44e2-a1a4-12778998a579',
  'title': 'Mi gran noche',
  'length': 184000},
 {'mbid': '5ac45a36-35d8-412f-8dd3-ebed8db96c97',
  'title': 'Cierro mis ojos',
  'length': 215000},
 {'mbid': '668c7810-46ea-4e3a-b988-416b57ba6ad1',
  'title': 'Hoy mejor que mañana',
  'length': 159000},
 {'mbid': 'adc357fd-4374-4fb2-99b7-76918c9a6921',
  'title': 'Mi hermano',
  'length': 144000},
 {'mbid': '37ee5438-78a1-4b75-a063-c6eb830462e7',
  'title': 'Un hombre vendrá',
  'length': 170000},
 {'mbid': 'f98f5eac-e272-4905-9368-ee7060d05fee',
  'title': 'Acuarela del río',
  'length': 175000},
 {'mbid': '5d26ebff-afe3-41c8-a464-3b31894a78df',
  'title': 'Tema de amor',
  'length': 152000},
 {'mbid': '88b4b817-44f9-47a8-839c-1c7f5888b054',
  'title': 'Llorona',
  'length': 230000},
 {'mbid': 'bcdf9808-b0ab-4310-a699-846b2bcef577',
  'title': 'Verano',
  'length': 138000},
 {'mbid

---

# Integrar datos MusicBrainz al grafo

Podemos utilizar dos opciones:
- Utilizar los datos (mbid, nombre, ...) obtenidos de wikidata
- Integrar directamente por nombre de autor, buscando en el api de MusicBrainz con el método ``search_artist_musicbrainz()``

El flujo es el siguiente:
Integrar información Artista >> Integrar información albumes >> Integrar información canciones

In [33]:
def add_musicbrainz_info_to_graph(artist_uri, artist_name, wikidata_mbid=None):
    if wikidata_mbid:
        mb_data = {
            'mbid': wikidata_mbid,
            'name': artist_name
        }
    else:
        mb_data = search_artist_musicbrainz(artist_name)
    
    if not mb_data or not mb_data.get('mbid'):
        return False
    
    artist_uri_ref = URIRef(artist_uri)
    mbid = mb_data['mbid']
    
    graph.add((artist_uri_ref, LOCAL.mbid, Literal(mbid)))
    
    mb_artist_uri = URIRef(f"https://musicbrainz.org/artist/{mbid}")
    graph.add((artist_uri_ref, OWL.sameAs, mb_artist_uri))
    
    if mb_data.get('type'):
        graph.add((artist_uri_ref, LOCAL.artistType, Literal(mb_data['type'])))
    
    if mb_data.get('country'):
        graph.add((artist_uri_ref, SCHEMA.nationality, Literal(mb_data['country'])))
    
    add_musicbrainz_albums_to_graph(mbid, artist_uri_ref, limit=5)


def add_musicbrainz_albums_to_graph(artist_mbid, artist_uri_ref, limit = 5):
    releases = get_releases_musicbrainz(artist_mbid)
    
    if not releases:
        return False

    for release in releases[:limit]:
        release_uri = URIRef(f"https://musicbrainz.org/release/{release['mbid']}")

        graph.add((release_uri, RDF.type, LOCAL.Album))
        graph.add((release_uri, RDF.type, MO.Record))
        graph.add((release_uri, DCTERMS.title, Literal(release['title'])))
        graph.add((release_uri, LOCAL.creator, artist_uri_ref))
        graph.add((release_uri, MO.maker, artist_uri_ref))
        graph.add((release_uri, LOCAL.mbid, Literal(release['mbid'])))
        
        if release.get('date'):
            try:
                date_str = release['date']
                if len(date_str) >= 10:
                    graph.add((release_uri, LOCAL.releaseDate, Literal(date_str[:10], datatype=XSD.date)))
            except:
                pass
        
        if release.get('country'):
            graph.add((release_uri, LOCAL.releaseCountry, Literal(release['country'])))
        
        add_musicbrainz_recordings_to_graph(release['mbid'], artist_uri_ref, limit=3)

        return True


def add_musicbrainz_recordings_to_graph(release_mbid, artist_uri_ref, limit = 3):
        recordings = get_recordings_from_release(release_mbid)
        if recordings:
            for recording in recordings[:limit]:
                recording_uri = URIRef(f"https://musicbrainz.org/recording/{recording['mbid']}")
                
                graph.add((recording_uri, RDF.type, LOCAL.Song))
                graph.add((recording_uri, RDF.type, MO.Track))
                graph.add((recording_uri, DCTERMS.title, Literal(recording['title'])))
                graph.add((recording_uri, LOCAL.creator, artist_uri_ref))
                
                if recording.get('length'):
                    duration_ms = int(recording['length'])
                    duration_seconds = duration_ms / 1000
                    graph.add((recording_uri, LOCAL.durationSeconds,
                                    Literal(duration_seconds, datatype=XSD.decimal)))
        
        return True

In [34]:
# Lo probamos con un solo artista

#primero añadimos sus datos de Wikidata
integrate_wikidata_artists([{
    'uri': 'http://www.wikidata.org/entity/Q180686',
    'name': 'Raphael',
    'genre': 'pop music',
    'country': 'Spain',
    'birthDate': '1943-05-05',
    'mbid': 'b34cfe02-994d-40dc-854f-bf54a23dd405'}])

add_musicbrainz_info_to_graph('http://www.wikidata.org/entity/Q180686', 'Raphael', 'b34cfe02-994d-40dc-854f-bf54a23dd405')

In [35]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_raphael.html")

out/grafo_ontologia_musical_con_raphael.html


In [37]:
# Lo probamos con todos los artistas obtenidos de Wikidata
for artist in artists:
    add_musicbrainz_info_to_graph(artist['uri'], artist['name'], artist.get('mbid'))
    time.sleep(1)  # Para no saturar el API

Error: ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))


In [38]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_artistas_wikidata.html")

out/grafo_ontologia_musical_con_artistas_wikidata.html


---

# Cargar Ontología Externa

## Carga una ontología externa con vocabulario adicional e integrarla.

### Utilizamos como ejemplo la ontología Music Ontology: http://musicontology.com/specification/index.html

**Proceso:**
1. Crea un grafo temporal para la ontología externa
2. Parsea el archivo/URL en el formato especificado
3. Fusiona los triples con el grafo principal

In [39]:
ontology_url = "http://purl.org/ontology/mo/"
external_graph = Graph()
external_graph.parse(ontology_url, format='xml')
for triple in external_graph:
    graph.add(triple)


## Añadimos los mapeos

In [40]:
graph.add((LOCAL[""], OWL.imports, URIRef("http://purl.org/ontology/mo/")))

mappings = [
    (LOCAL.MusicalArtist, MO.MusicArtist, "Artista Musical"),
    (LOCAL.Album, MO.Record, "Álbum"),
    (LOCAL.Song, MO.Track, "Canción"),
]

for local_class, mo_class, label in mappings:
    graph.add((local_class, OWL.equivalentClass, mo_class))
    graph.add((local_class, RDFS.label, Literal(label, lang="es")))

In [41]:
#Vamos a ver cómo queda
sbc.show_graph(graph, output_file="grafo_ontologia_musical_con_mo.html")

out/grafo_ontologia_musical_con_mo.html


### Guardar la ontología

In [42]:
sbc.save(graph, "grafo_ontologia_musical.ttl", format='turtle')

Ontología guardada en: data\grafo_ontologia_musical.ttl


---

# Consultas sobre la Ontología Integrada

## Ejemplo de consulta SPARQL sobre la ontología integrada

**Objetivo:** Encontrar todos los artistas y sus álbumes en la ontología

In [43]:
query1 = f"""
PREFIX local: <{LOCAL}>
        PREFIX rdfs: <{RDFS}>
        PREFIX dcterms: <{DCTERMS}>

SELECT ?artist ?artistName ?album ?albumTitle WHERE {{
    ?artist a local:SoloArtist ;
            rdfs:label ?artistName .
     ?album local:creator ?artist ;
            dcterms:title ?albumTitle .
}}
"""

print("Artistas con sus álbumes:")
results = graph.query(query1)
for row in results:
    artist = row.artistName
    album = row.albumTitle
    print(f"- {artist} - Álbum: {album}")

    

Artistas con sus álbumes:
- Myleene Klass - Álbum: My Lullabies: Motown
- Myleene Klass - Álbum: Three Times a Lady
- Myleene Klass - Álbum: What Becomes of the Brokenhearted?
- Myleene Klass - Álbum: Just My Imagination
- Jasmin Tabatabai - Álbum: 101 Nacht
- Jasmin Tabatabai - Álbum: Prolog, Teil 1
- Jasmin Tabatabai - Álbum: Prolog, Teil 2
- Jasmin Tabatabai - Álbum: Prolog, Teil 3
- Corinna May - Álbum: I Can't Live Without Music
- Corinna May - Álbum: I Can't Live Without Music
- Corinna May - Álbum: Can't Go To Sleep Without You
- Corinna May - Álbum: Light My Fire
- Nolwenn Leroy - Álbum: Histoires naturelles
- Nolwenn Leroy - Álbum: Nolwenn Ohwo !
- Nolwenn Leroy - Álbum: L’Enfant cerf‐volant
- Nolwenn Leroy - Álbum: Reste encore
- Raphael - Álbum: El ángel
- Raphael - Álbum: El ángel
- Raphael - Álbum: Corazón, corazón
- Raphael - Álbum: Madre


# Ejercicio

## Crea un grafo de conocimiento para el dominio que consideres más oportuno integrando conocimiento de varias fuentes de conocimiento semántico.